# GVP-EGNN Unified Training (Pipeline C v2)

**Architecture upgrades over v1:**
- GVP-enhanced EGNN (direction vectors in message passing)
- 6 layers, hidden_dim=96, 8 vector channels
- Three simultaneous losses: NSF flow + vMF direction + Gaussian energy
- Auxiliary loss schedule (strong early, fade late)

**Runtime:** Select **A100 GPU** for full training (~20 CU), or **T4 GPU** for smoke test (~1 CU).

**Before running:** Push latest code to GitHub and upload data to Google Drive.

In [ ]:
# 1. Check GPU & Install Dependencies
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('ERROR: Not connected to a GPU!')
    print('Go to Runtime > Change runtime type > A100 GPU (or T4 for smoke test)')
else:
    print(gpu_info)
    print('\nGPU is ready!')

In [ ]:
!pip install sbi tensorboard tqdm psutil -q
print('Dependencies installed!')

## 2. Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

# Reset to safe directory first (in case previous cwd was deleted)
os.chdir('/content')

REPO_DIR = '/content/sbi-srim'
BRANCH = 'feature/gvp-egnn'

# Clean clone to ensure correct branch
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('Removed old repo clone.')

!git clone -b {BRANCH} https://github.com/cbharathulwar/sbi-srim.git {REPO_DIR}
print(f'Repo cloned (branch: {BRANCH})!')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!git branch
!git log --oneline -5

In [ ]:
# Copy data from Google Drive to local storage (faster I/O)
DRIVE_DATA_DIR = '/content/drive/MyDrive/sbi-srim-data'
LOCAL_DATA_DIR = f'{REPO_DIR}/data/mcpe3d'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

for csv_name in ['mcpe_3d_train.csv', 'mcpe_3d_eval.csv']:
    src = os.path.join(DRIVE_DATA_DIR, csv_name)
    dst = os.path.join(LOCAL_DATA_DIR, csv_name)
    if os.path.exists(dst):
        print(f'  {csv_name} already exists locally')
    elif os.path.exists(src):
        print(f'  Copying {csv_name}...')
        shutil.copy2(src, dst)
        print(f'  Done! ({os.path.getsize(dst) / 1e6:.1f} MB)')
    else:
        print(f'  ERROR: {src} not found! Upload to Google Drive: sbi-srim-data/')

# Copy preprocessing cache if available
for cache_file in glob.glob(os.path.join(DRIVE_DATA_DIR, '*.egnn_cache.pt')):
    cache_name = os.path.basename(cache_file)
    dst = os.path.join(LOCAL_DATA_DIR, cache_name)
    if not os.path.exists(dst):
        print(f'  Copying cache: {cache_name}...')
        shutil.copy2(cache_file, dst)

# Copy previous GVP checkpoint if resuming
DRIVE_GVP_DIR = '/content/drive/MyDrive/sbi-srim-results/gvp_egnn'
LOCAL_GVP_DIR = f'{REPO_DIR}/results/gvp_egnn'
os.makedirs(LOCAL_GVP_DIR, exist_ok=True)

for ckpt_name in ['checkpoint.pt', 'best_checkpoint.pt']:
    src = os.path.join(DRIVE_GVP_DIR, ckpt_name)
    dst = os.path.join(LOCAL_GVP_DIR, ckpt_name)
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copying {ckpt_name} from previous session...')
        shutil.copy2(src, dst)

print('\nData files:')
!ls -lh {LOCAL_DATA_DIR}/

## 3. Smoke Test (T4 GPU, ~5 min)
Run 5 epochs on 5k tracks to verify everything works. Skip if going straight to full training.

In [ ]:
# Verify we're in the right place and the script exists
import os
os.chdir('/content/sbi-srim')
print(f'cwd: {os.getcwd()}')
!ls src/scripts/
!echo "---"
!python -m src.scripts.train_gvp_egnn --smoke

## 4. Full Training (A100 GPU, ~2-4 hours)
200 epochs on 100k tracks. Early stopping with patience=40.

In [ ]:
# Full training run (use --resume to continue from checkpoint)
import os
os.chdir('/content/sbi-srim')
!python -m src.scripts.train_gvp_egnn

## 5. Save Results to Google Drive

In [ ]:
# Save results to Google Drive
DRIVE_GVP_DIR = '/content/drive/MyDrive/sbi-srim-results/gvp_egnn'
LOCAL_GVP_DIR = f'{REPO_DIR}/results/gvp_egnn'

if os.path.exists(LOCAL_GVP_DIR):
    os.makedirs(DRIVE_GVP_DIR, exist_ok=True)
    for f in os.listdir(LOCAL_GVP_DIR):
        src = os.path.join(LOCAL_GVP_DIR, f)
        dst = os.path.join(DRIVE_GVP_DIR, f)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1e6
            print(f'  Saved: {f} ({size_mb:.1f} MB)')
    print(f'\nAll results saved to: {DRIVE_GVP_DIR}')
else:
    print('No results found.')

## 6. View Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

LOCAL_GVP_DIR = f'{REPO_DIR}/results/gvp_egnn'

# Show eval summary
eval_csv = f'{LOCAL_GVP_DIR}/eval_results.csv'
if os.path.exists(eval_csv):
    df = pd.read_csv(eval_csv)
    print(f'Evaluated {len(df)} tracks\n')
    print(f'Median Energy Error:   {df["energy_error"].median():.2f} keV')
    print(f'Median Angular Error:  {df["angular_error_deg"].median():.2f} deg')
    print(f'Head-Tail Accuracy:    {(1 - (df["angular_error_deg"] > 90).mean()) * 100:.1f}%')
    if 'energy_std' in df.columns:
        print(f'Median Energy sigma:   {df["energy_std"].median():.2f} keV')
    if 'angular_cone_68' in df.columns:
        print(f'Median 68% Cone:       {df["angular_cone_68"].median():.2f} deg')

# Show training curves
train_log = f'{LOCAL_GVP_DIR}/training_log.csv'
if os.path.exists(train_log):
    df_log = pd.read_csv(train_log)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Total loss
    axes[0,0].plot(df_log['epoch'], df_log['train_total'], label='Train')
    axes[0,0].plot(df_log['epoch'], df_log['val_total'], label='Val')
    axes[0,0].set_title('Total Loss')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)

    # Flow loss
    axes[0,1].plot(df_log['epoch'], df_log['train_flow'], label='Train')
    axes[0,1].plot(df_log['epoch'], df_log['val_flow'], label='Val')
    axes[0,1].set_title('Flow Loss (SBI)')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)

    # Direction loss
    axes[1,0].plot(df_log['epoch'], df_log['train_dir'], label='Train')
    axes[1,0].plot(df_log['epoch'], df_log['val_dir'], label='Val')
    axes[1,0].set_title('Direction Loss (vMF)')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)

    # Energy loss
    axes[1,1].plot(df_log['epoch'], df_log['train_energy'], label='Train')
    axes[1,1].plot(df_log['epoch'], df_log['val_energy'], label='Val')
    axes[1,1].set_title('Energy Loss (Gaussian)')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)

    for ax in axes.flat:
        ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()

# Show saved plots
import glob
for pf in sorted(glob.glob(f'{LOCAL_GVP_DIR}/*.png')):
    print(f'\n--- {os.path.basename(pf)} ---')
    display(Image(filename=pf, width=800))